Layered Azure Machine Learning Pipeline (SDK v2)
================================================


Create a layered pipeline in Azure ML and publish it.

Write a simple Python code snippet to run it, including descriptions of the input and output and related details.
Just submit the Python code file.

### Step 1: Initialize Azure ML Client
This cell initializes the `MLClient` using `DefaultAzureCredential` to connect to your Azure Machine Learning Workspace. Make sure to replace the placeholder values for subscription, resource group, and workspace with your actual Azure resource details.

In [16]:
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient

# Replace these with your actual Azure resource details
subscription_id = "YOUR_SUBSCRIPTION_ID"
resource_group = "YOUR_RESOURCE_GROUP"
workspace_name = "YOUR_WORKSPACE_NAME"

# Initialize the MLClient
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name
)

print("ml_client successfully initialized.")

ml_client successfully initialized.


### Step 2: Define and Construct the Layered Pipeline
In this step, we define the reusable dummy command components (`Prepare Data` and `Train Model`) and orchestrate them into a unified pipeline workflow using the `@dsl.pipeline` decorator.

In [17]:
"""
Toy layered Azure ML pipeline (SDK v2).

Assumptions:
- ml_client is authenticated for the target Azure ML workspace.
- "cpu-cluster" already exists.

Layer 1: clean raw data.
Layer 2: train a model using the cleaned data.
Output: the model artifact from Layer 2.
"""

from azure.ai.ml import Input, Output, dsl, command
from azure.core.exceptions import ClientAuthenticationError

# Define dummy prep_component to resolve NameError
prep_component = command(
    name="prep_data_component",
    display_name="Prepare Data",
    description="A dummy component to clean raw data",
    inputs={"data": Input(type="uri_folder")},
    outputs={"clean_data": Output(type="uri_folder")},
    # Simulated execution command
    code=".",
    command="echo 'Preparing data...' && mkdir -p ${{outputs.clean_data}} && cp -r ${{inputs.data}}/* ${{outputs.clean_data}}/ || true",
    environment="azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
)

# Define dummy train_component to resolve NameError
train_component = command(
    name="train_model_component",
    display_name="Train Model",
    description="A dummy component to train a model",
    inputs={"clean_data": Input(type="uri_folder")},
    outputs={"model": Output(type="uri_folder")},
    # Simulated execution command
    code=".",
    command="echo 'Training model...' && mkdir -p ${{outputs.model}} && echo 'my_dummy_model' > ${{outputs.model}}/model.txt",
    environment="azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
)

@dsl.pipeline(
    name="toy-layered-training-pipeline",
    description="Prepare data, then train a toy model.",
    default_compute="cpu-cluster",
)
def training_pipeline(raw_data):
    # Layer 1: prepare the data
    prep = prep_component(data=raw_data)

    # Layer 2: train using Layer 1's output
    train = train_component(
        clean_data=prep.outputs.clean_data
    )

    # Return the final model artifact
    return {"model": train.outputs.model}


# Use an existing Azure ML data asset
pipeline_job = training_pipeline(
    raw_data=Input(
        type="uri_folder",
        path="azureml:bank-data:1"
    )
)

try:
    # Submit the pipeline job
    submitted_job = ml_client.jobs.create_or_update(
        pipeline_job,
        experiment_name="toy-layered-training",
    )
    print("Pipeline submitted successfully.")
    print("Job name:", submitted_job.name)
    print("Studio URL:", submitted_job.studio_url)
except ClientAuthenticationError as auth_err:
    print("ClientAuthenticationError caught!")
    print("Reason: Active credentials to Azure ML Workspace are not configured in this local Colab environment.")
    print("To run this pipeline successfully, authenticate using the Azure CLI (`az login`) or configure your environment variables.")
    print(f"Details: {auth_err}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	WorkloadIdentityCredential: WorkloadIdentityCredential authentication unavailable. The workload options are not fully configured. See the troubleshooting guide for more information: https://aka.ms/azsdk/python/identity/workloadidentitycredential/troubleshoot. Missing required arguments: 'tenant_id', 'client_id', 'token_file_path'.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	VisualStudioCodeCredential: VisualStudioCodeCredential requires the 'azure-identity-broker' package to be installed. You must also ensure you have the Azure Resources extension installed and have signed in to Az

ClientAuthenticationError caught!
Reason: Active credentials to Azure ML Workspace are not configured in this local Colab environment.
To run this pipeline successfully, authenticate using the Azure CLI (`az login`) or configure your environment variables.
Details: DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	WorkloadIdentityCredential: WorkloadIdentityCredential authentication unavailable. The workload options are not fully configured. See the troubleshooting guide for more information: https://aka.ms/azsdk/python/identity/workloadidentitycredential/troubleshoot. Missing required arguments: 'tenant_id', 'client_id', 'token_file_path'.
	ManagedIdentityCredential: ManagedIdentityCredential authentication

### Step 3: Write Pipeline Source Code to a Local Script File
Since active Azure authentication is not configured in this temporary notebook environment (which results in a `ClientAuthenticationError`), we write the complete pipeline setup and execution code into a local standalone script file named `azure_ml_pipeline_toy.py`. You can download this script and execute it locally or within your secured terminal where you have logged in via Azure CLI (`az login`).

In [18]:
with open("azure_ml_pipeline_toy.py", "w", encoding="utf-8") as f:
    f.write('''# -*- coding: utf-8 -*-
"""
Complete Azure ML pipeline (SDK v2) script.
Save this script as `azure_ml_pipeline_toy.py` and run it in an environment
where you are authenticated with Azure (e.g., via Azure CLI `az login`).
"""

from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient, Input, Output, dsl, command

# 1. Initialize ML Client
# Replace these placeholder values with your actual Azure resource details
subscription_id = "YOUR_SUBSCRIPTION_ID"
resource_group = "YOUR_RESOURCE_GROUP"
workspace_name = "YOUR_WORKSPACE_NAME"

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name
)

# 2. Define Components
prep_component = command(
    name="prep_data_component",
    display_name="Prepare Data",
    description="A component to clean raw data",
    inputs={"data": Input(type="uri_folder")},
    outputs={"clean_data": Output(type="uri_folder")},
    code=".",
    command="echo 'Preparing data...' && mkdir -p ${{outputs.clean_data}} && cp -r ${{inputs.data}}/* ${{outputs.clean_data}}/ || true",
    environment="azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
)

train_component = command(
    name="train_model_component",
    display_name="Train Model",
    description="A component to train a model",
    inputs={"clean_data": Input(type="uri_folder")},
    outputs={"model": Output(type="uri_folder")},
    code=".",
    command="echo 'Training model...' && mkdir -p ${{outputs.model}} && echo 'my_dummy_model' > ${{outputs.model}}/model.txt",
    environment="azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
)

# 3. Define Pipeline
@dsl.pipeline(
    name="toy-layered-training-pipeline",
    description="Prepare data, then train a toy model.",
    default_compute="cpu-cluster",
)
def training_pipeline(raw_data):
    # Layer 1: prepare the data
    prep = prep_component(data=raw_data)

    # Layer 2: train using Layer 1's output
    train = train_component(
        clean_data=prep.outputs.clean_data
    )

    # Return the final model artifact
    return {"model": train.outputs.model}

# 4. Instantiate and Submit Job
# Use an existing data asset on your Workspace (e.g., bank-data version 1)
pipeline_job = training_pipeline(
    raw_data=Input(
        type="uri_folder",
        path="azureml:bank-data:1"
    )
)

try:
    submitted_job = ml_client.jobs.create_or_update(
        pipeline_job,
        experiment_name="toy-layered-training",
    )
    print("Pipeline submitted successfully!")
    print("Job name:", submitted_job.name)
    print("Studio URL:", submitted_job.studio_url)
except Exception as e:
    print("Error submitting job (Verify your credentials and Azure resources):")
    print(e)
''')

print("Successfully saved pipeline script to file: azure_ml_pipeline_toy.py")

Successfully saved pipeline script to file: azure_ml_pipeline_toy.py
